# Toy example — min-cost-flow

A tiny 4-country demo so you can verify the API without downloading FAOSTAT.

Two countries produce more than they need, two countries need more than they produce.
We tell the model the cost of every possible trade route and let it pick the cheapest feasible set of flows.
See `docs/methodology.md` for the full LP formulation.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.model import FertilizerMCF
from src.postprocessing import summary_table


## 1. Inputs

`supply` and `demand` are in tonnes. `edges` lists the allowed trade routes with unit cost and capacity.

In [ ]:
countries = ['A', 'B', 'C', 'D']
supply = pd.Series({'A': 100, 'B':  80, 'C':  20, 'D':  30})
demand = pd.Series({'A':  40, 'B':  30, 'C':  70, 'D':  90})

edges = pd.DataFrame([
    ('A', 'C', 1.0, 1000),
    ('A', 'D', 2.5, 1000),
    ('B', 'C', 1.5, 1000),
    ('B', 'D', 0.8, 1000),
    ('A', 'B', 0.2, 1000),
], columns=['from', 'to', 'cost', 'capacity'])
edges


## 2. Solve

In [ ]:
model = FertilizerMCF(supply, demand, edges)
result = model.solve()
print('Feasible :', result.feasible)
print('Flow     :', round(result.total_flow, 1))
print('Cost     :', round(result.total_cost, 2))
result.flow_matrix


## 3. Per-country summary

`supply + imports - exports = availability`. Coverage = `availability / demand`.


In [ ]:
summary_table(result)

## 4. Quick visual

In [ ]:
from src.utils import plot_flow_sankey
fig = plot_flow_sankey(result, title='Toy flows')
fig.show()
